# D3 · 99 · Comparison of the Dataset 3 experiments (saved outputs only)

Subject-level results on Dataset 3 (ds004504, 36 AD / 29 HC, within-D3 cross-validation): EARM-GNN and its two
ablations, the multi-band GCN, source and sensor PSD models and the age-only baseline. Tables, ROC curves,
confusion matrices and per-repeat paired differences (descriptive; no significance test). Nothing is trained.
Run after the D3 experiment notebooks; missing experiments are simply absent.
**How to use:** *Runtime → Run all*. Nothing to edit. Allow Google Drive access when asked.

If Colab disconnects or the runtime ends, open the notebook again and *Run all* again: finished
work is kept on Google Drive and skipped, and the computation continues where it stopped.
Everything (checkpoints, results, figures, manifests) is written to Drive, never to the
temporary Colab disk.

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_CODE = False   # leave False. True only if you were asked to switch to newer code
                      # (all notebooks then use the newest commit; unfinished work is recomputed).
BRANCH = 'claude/ecstatic-davinci-a2e8mh'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
print('outputs on Drive:', OUT)

## 0. Dataset 3 code pin
All Dataset 3 notebooks run one commit, recorded in `<outputs>/D3_CODE_COMMIT.txt` on the first D3 run; the
D1/D2 experiments keep their own pin (`CODE_COMMIT.txt`, unchanged). Leave `UPDATE_D3_CODE = False`.

In [ ]:
# Dataset 3 code pin: every D3 notebook (download, graphs, experiments) runs the commit recorded in
# <outputs>/D3_CODE_COMMIT.txt (set on the first D3 run), independent of the D1/D2 pin in CODE_COMMIT.txt.
UPDATE_D3_CODE = False   # leave False; True moves only the D3 pin to the newest code
import subprocess, sys
from pathlib import Path
from adgraph.colab import resolve_code_pin

head = subprocess.check_output(['git', '-C', '/content/repo', 'rev-parse', f'origin/{BRANCH}'], text=True).strip()
d3_work = (any(Path(OUT).glob('graphsD3_v*/subjects/*/subject.done.json'))
           or any(Path(OUT).glob('experiments_D3/**/*.done.json')))
D3_COMMIT, _ = resolve_code_pin(Path(OUT) / 'D3_CODE_COMMIT.txt', head, update=UPDATE_D3_CODE, has_work=d3_work)
_run(['git', '-C', '/content/repo', 'checkout', '-q', D3_COMMIT])
for f in ('src/adgraph/io/bids.py', 'configs/pipeline_d3.yaml', 'scripts/d3_data.py'):
    if not Path('/content/repo', f).exists():
        raise RuntimeError(f'D3 pin {D3_COMMIT[:10]} lacks {f}: set UPDATE_D3_CODE = True and run again')
_run([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/content/repo/requirements.txt'])
for m in [m for m in sys.modules if m == 'adgraph' or m.startswith('adgraph.')]:
    del sys.modules[m]                      # this notebook's own imports now come from the D3 commit
from adgraph.colab import choose_output_root, find_data_root, sh, write_paths_yaml
from adgraph.config import load_paths
_root = find_data_root()
write_paths_yaml('/content/repo', _root, choose_output_root(_root))   # adds the Dataset 3 location
PATHS = load_paths('/content/repo/configs/paths.yaml')
print('D3 code commit:', D3_COMMIT)
print('D1/D2 pin (unchanged):', (Path(OUT) / 'CODE_COMMIT.txt').read_text().split()[0])
print('Dataset 3 folder:', PATHS['dataset3']['root'])

## 1. Comparison

In [ ]:
sh("python scripts/exp7_earm.py compare --family D3")
import pandas as pd
from IPython.display import Image, display
cmp = sorted((Path(OUT) / 'experiments_D3' / '_comparison').glob('comparison_*'))[-1]
display(pd.read_csv(cmp / 'auc_by_setting.csv', index_col=0).round(3))
t = pd.read_csv(cmp / 'comparison_table.csv') if (cmp / 'comparison_table.csv').stat().st_size > 1 else pd.DataFrame()
if len(t):
    display(t[['model', 'connectivity', 'n_repeats', 'roc_auc_mean', 'roc_auc_sd', 'roc_auc_ci95_low',
               'roc_auc_ci95_high', 'balanced_accuracy_mean', 'sensitivity_mean', 'specificity_mean', 'f1_mean',
               'n_parameters', 'fit_seconds_mean']].round(3))
for png in sorted((cmp / 'figures').glob('D3_*.png')):
    print(png.name)
    display(Image(filename=str(png)))